In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
import sys
sys.path.append('../')


In [2]:
%load_ext autoreload
%autoreload 2
import os
import pandas as pd
from loglizer.models import PCA, IsolationForest, DecisionTree, LR
from loglizer import dataloader, preprocessing
import pickle
from tqdm import tqdm
from sklearn.ensemble import IsolationForest as iForest
from sklearn.metrics import precision_recall_fscore_support


def get_x_y(windows, content2tempalte):
    x = []
    y = []
    for window in tqdm(windows):
        template_list = []
        y_list = []
        for item in window:
            template = content2tempalte.get(item["Content"], "")
            template_list.append(template)
            y_list.append(item["Label"])
        x.append(template_list)
        y.append(1 if sum(y_list) > 0 else 0)
    return x, y 


ModuleNotFoundError: No module named 'pandas'

In [ ]:
dataname = "BGL"

config_info = {
    "BGL": {
        "structure_file": "../../data/BGL/BGL.log_structured.csv",
        "pkl_path": "../../proceeded_data/BGL/BGL_ws=60m_s=60m_0.8train",
    },
    "Thunderbird": {
        "structure_file": "../../data/Thunderbird/Thunderbird_10m.log_structured_drain.csv",
        "pkl_path": "../../proceeded_data/Thunderbird/Thunderbird_ws=60m_s=60m_0.8train",
    },
}

structure_file = config_info[dataname]["structure_file"]
pkl_path = config_info[dataname]["pkl_path"]

In [ ]:
parsed_result = pd.read_csv(structure_file)
content2tempalte = dict(zip(parsed_result["Content"], parsed_result["EventTemplate"]))

In [ ]:
parsed_result["EventTemplate"].nunique()

In [ ]:
with open(os.path.join(pkl_path, "train.pkl"), "rb") as fr:
    train_windows = pickle.load(fr)
with open(os.path.join(pkl_path, "test.pkl"), "rb") as fr:
    test_windows = pickle.load(fr)

train_x, train_y = get_x_y(train_windows, content2tempalte)
test_x, test_y = get_x_y(test_windows, content2tempalte)

In [ ]:
import numpy as np
feature_extractor = preprocessing.FeatureExtractor()
train_feat = feature_extractor.fit_transform(np.array(train_x), term_weighting='tf-idf', normalization='zero-mean')

In [ ]:
print("train shape: {}".format(train_feat.shape))

In [ ]:
model_name = "dt"

if model_name.lower() == "if":
    model = iForest(n_estimators=100, max_features=1)
    model.fit(train_feat)

    pred_train = model.decision_function(train_feat)
    proba_train = (pred_train-pred_train.min()) / (pred_train.max() - pred_train.min())
    
    test_feat = feature_extractor.transform(np.array(test_x))
    pred_test = model.decision_function(test_feat)
    proba_test = (pred_test-pred_test.min()) / (pred_test.max() - pred_test.min())

elif model_name.lower() == "dt":
    model = DecisionTree()
    model.fit(train_feat, train_y)

    proba_train = model.predict_proba(train_feat)[:, 1]
    
    test_feat = feature_extractor.transform(np.array(test_x))
    proba_test = model.predict_proba(test_feat)[:, 1]
    
elif model_name.lower() == "lr":
    model = LR()
    model.fit(train_feat, train_y)
    proba_train = model.predict_proba(train_feat)[:, 1]

    test_feat = feature_extractor.transform(np.array(test_x))
    proba_test = model.predict_proba(test_feat)[:, 1]

In [ ]:
th = 0.15 # for BGL, lr
th = 0.99 # for BGL, if


for th in np.arange(0.01, 1, 0.05):
    # pred_train = (proba_train>th).astype(int)
    # precision, recall, f1, _ = precision_recall_fscore_support(train_y, pred_train, average='binary')
    # print("Train accuracy:")
    # print('Precision: {:.3f}, recall: {:.3f}, F1-measure: {:.3f}\n'.format(precision, recall, f1))
    pred_test = (proba_test>th).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(test_y, pred_test, average='binary')
    print("Test accuracy:")
    print('Precision: {:.3f}, recall: {:.3f}, F1-measure: {:.3f}\n'.format(precision, recall, f1))